## Bloque 0 — Importación de librerías

**Qué hace:** carga todas las dependencias del notebook.

**Para qué sirve:** reúne en un solo lugar las herramientas que se usan después. `hashlib` genera los hashes SHA-256; `json` permite la serialización estable; `time` entrega los timestamps; `dataclasses` y `typing` modelan el bloque; `pandas` arma la tabla de auditoría; `scikit-learn` y `joblib` entrenan y guardan el modelo real; `os` crea la carpeta de artefactos y `deepcopy` permite alterar una copia de la cadena sin dañar la original.

In [1]:
import hashlib
import json
import time
from dataclasses import dataclass, asdict, field
from typing import Any
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
import joblib
import os
from copy import deepcopy

## Bloque 1 — Estructura del bloque y su hash

**Qué hace:** define la clase `Bloque` (índice, timestamp, datos, hash_previo y nonce) y la función `hash_bloque`, que calcula el hash de un bloque sobre una representación canónica.

**Para qué sirve:** es la unidad mínima de registro. El `hash_previo` es lo que enlazará cada bloque con el anterior. El uso de `sort_keys=True` garantiza que el mismo contenido lógico produzca siempre el mismo hash, sin importar el orden de las claves del diccionario.

In [2]:
@dataclass
class Bloque:
    indice: int
    timestamp: float
    datos: dict[str, Any]
    hash_previo: str
    nonce: int = 0

def hash_bloque(bloque: Bloque) -> str:
    """
    Hash del bloque sobre una representación CANÓNICA.
    sort_keys=True evita que dos máquinas produzcan hashes distintos
    para el mismo contenido lógico.
    """
    payload = asdict(bloque)
    texto = json.dumps(payload, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(texto.encode()).hexdigest()

## Bloque 2 — Cadena y bloque génesis

**Qué hace:** crea la lista `cadena`, agrega el bloque **génesis** (el primero, con `hash_previo = "0"`) y define `nuevo_bloque`, que añade bloques enlazados al hash del último.

**Para qué sirve:** inaugura la historia de la cadena y entrega el mecanismo para ir agregando eventos. Cada bloque nuevo apunta criptográficamente al anterior, formando la cadena de evidencias.

In [3]:
cadena: list[Bloque] = []

# Bloque génesis: inaugura la cadena del pipeline
genesis = Bloque(
    indice=0,
    timestamp=time.time(),
    datos={"msg": "inicio_auditoria_pipeline_ml"},
    hash_previo="0",
)
cadena.append(genesis)

def nuevo_bloque(datos: dict) -> None:
    """Agrega un bloque enlazado al hash del último bloque existente."""
    previo = cadena[-1]
    hprev = hash_bloque(previo)
    bloque = Bloque(
        indice=len(cadena),
        timestamp=time.time(),
        datos=datos,
        hash_previo=hprev,
    )
    cadena.append(bloque)

## Bloque 3 — Pipeline de Machine Learning real

**Qué hace:** ejecuta un pipeline de ML completo sobre el dataset **Iris** y registra cada etapa como un bloque de la cadena. Los cinco pasos son: ingesta del dataset bruto, limpieza, generación de features, entrenamiento de un Random Forest y validación con métricas reales (accuracy y F1). Cada paso guarda un archivo real en la carpeta `artefactos/` y lo hashea con `hash_archivo`.

**Para qué sirve:** es el corazón de la actividad. Demuestra el **linaje verificable** del pipeline: la salida de cada etapa (su hash) se convierte en la entrada de la siguiente, dejando evidencia de toda la historia de transformaciones del dato.

In [4]:
# Carpeta donde guardamos los artefactos reales del pipeline
os.makedirs("artefactos", exist_ok=True)

def hash_archivo(ruta: str) -> str:
    """Hash SHA-256 de un archivo REAL leído en binario."""
    with open(ruta, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()

# ---- Paso 1: dataset bruto (origen) ----
X, y = load_iris(return_X_y=True, as_frame=True)
dataset_bruto = X.copy()
dataset_bruto["target"] = y
dataset_bruto.to_csv("artefactos/iris_bruto.csv", index=False)
h_bruto = hash_archivo("artefactos/iris_bruto.csv")
nuevo_bloque({
    "paso": "ingesta_dataset_bruto",
    "actor": "pipeline_etl",
    "input": None,
    "output": h_bruto,
})

# ---- Paso 2: limpieza (quitar nulos y duplicados) ----
dataset_limpio = dataset_bruto.dropna().drop_duplicates()
dataset_limpio.to_csv("artefactos/iris_limpio.csv", index=False)
h_limpio = hash_archivo("artefactos/iris_limpio.csv")
nuevo_bloque({
    "paso": "limpieza",
    "actor": "notebook_07",
    "input": h_bruto,
    "output": h_limpio,
})

# ---- Paso 3: feature engineering (separar X / y para entrenar) ----
features = dataset_limpio.drop(columns=["target"])
etiquetas = dataset_limpio["target"]
features.to_csv("artefactos/iris_features.csv", index=False)
h_features = hash_archivo("artefactos/iris_features.csv")
nuevo_bloque({
    "paso": "features",
    "actor": "notebook_07",
    "input": h_limpio,
    "output": h_features,
})

# ---- Paso 4: entrenamiento REAL del modelo ----
X_train, X_test, y_train, y_test = train_test_split(
    features, etiquetas, random_state=42
)
modelo = RandomForestClassifier(n_estimators=100, random_state=42)
modelo.fit(X_train, y_train)
joblib.dump(modelo, "artefactos/modelo_rf.pkl")
h_modelo = hash_archivo("artefactos/modelo_rf.pkl")
nuevo_bloque({
    "paso": "entrenamiento",
    "actor": "trainer_script",
    "input": h_features,
    "output": h_modelo,
})

# ---- Paso 5: validación / reporte de métricas REALES ----
pred = modelo.predict(X_test)
acc = accuracy_score(y_test, pred)
f1 = f1_score(y_test, pred, average="macro")
reporte = {"accuracy": round(acc, 4), "f1_macro": round(f1, 4)}
with open("artefactos/metricas.json", "w") as f:
    json.dump(reporte, f, sort_keys=True)
h_reporte = hash_archivo("artefactos/metricas.json")
nuevo_bloque({
    "paso": "validacion_reporte",
    "actor": "trainer_script",
    "input": h_modelo,
    "output": h_reporte,
})

print(f"Cadena construida con {len(cadena)} bloques (génesis + 5 pasos).")
print(f"Métricas reales del modelo: accuracy={acc:.4f}, f1_macro={f1:.4f}")

Cadena construida con 6 bloques (génesis + 5 pasos).
Métricas reales del modelo: accuracy=1.0000, f1_macro=1.0000


## Bloque 4 — Función de validación

**Qué hace:** recorre la cadena desde el segundo bloque y verifica que el `hash_previo` de cada bloque coincida con el hash recalculado del bloque anterior. Devuelve `False` ante la primera inconsistencia.

**Para qué sirve:** es el invariante central del entregable. Permite comprobar la integridad de la cadena sin confiar en quién la entrega: basta con ejecutar las mismas reglas de verificación.

In [5]:
def validar_cadena(cadena: list[Bloque]) -> bool:
    """
    Verifica que cada referencia hash_previo coincida con el hash
    real del bloque anterior. Devuelve False ante la primera ruptura.
    """
    for i in range(1, len(cadena)):
        actual = cadena[i]
        previo = cadena[i - 1]
        if actual.hash_previo != hash_bloque(previo):
            print(f"  ✗ Ruptura detectada en el bloque {i} "
                  f"(paso: {actual.datos.get('paso', '—')})")
            return False
    return True

print("¿Cadena válida tras construcción?", validar_cadena(cadena))  # True

¿Cadena válida tras construcción? True


## Bloque 5 — Tabla pandas de auditoría

**Qué hace:** convierte cada bloque en una fila de un `DataFrame` con índice, paso, actor, timestamp, hashes (entrada, salida, previo y del bloque) y el resultado de la validación. Guarda la tabla como CSV en `artefactos/`.

**Para qué sirve:** es el entregable de la tabla pandas que pide la diapositiva. Presenta toda la historia del pipeline de forma legible y deja una evidencia exportada que puede auditarse después.

In [6]:
# Tabla pandas con eventos, hashes, timestamps y resultado de validación
os.makedirs("artefactos", exist_ok=True)

def recortar_hash(valor):
    """
    Recorta hashes largos para que la tabla sea más legible.
    Si el valor no existe, devuelve "—".
    """
    if valor is None:
        return "—"
    if isinstance(valor, str) and len(valor) > 16:
        return valor[:16] + "..."
    return valor


cadena_valida = validar_cadena(cadena)

filas = []

for b in cadena:
    filas.append({
        "indice": b.indice,
        "paso": b.datos.get("paso", b.datos.get("msg", "—")),
        "actor": b.datos.get("actor", "—"),
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(b.timestamp)),
        "hash_input": recortar_hash(b.datos.get("input")),
        "hash_output": recortar_hash(b.datos.get("output")),
        "hash_previo": recortar_hash(b.hash_previo),
        "hash_bloque": recortar_hash(hash_bloque(b)),
        "resultado_validacion": "válida" if cadena_valida else "inválida",
    })

df = pd.DataFrame(filas)

# Guardamos la tabla como evidencia del entregable
df.to_csv("artefactos/tabla_eventos_blockchain.csv", index=False)

df

,indice,paso,actor,timestamp,hash_input,hash_output,hash_previo,hash_bloque,resultado_validacion
0,0,inicio_auditoria_pipeline_ml,—,2026-06-24 12:30:19,—,—,0,eab9a39c75d7e0a3...,válida
1,1,ingesta_dataset_bruto,pipeline_etl,2026-06-24 12:30:19,—,0ba79ae755c686ee...,eab9a39c75d7e0a3...,325e36115ffaacfb...,válida
2,2,limpieza,notebook_07,2026-06-24 12:30:19,0ba79ae755c686ee...,f314cc94fd6dab9f...,325e36115ffaacfb...,e9141efb282dc074...,válida
3,3,features,notebook_07,2026-06-24 12:30:19,f314cc94fd6dab9f...,3c4445a5a61e644a...,e9141efb282dc074...,8cd35b1e27159168...,válida
4,4,entrenamiento,trainer_script,2026-06-24 12:30:19,3c4445a5a61e644a...,91d56bfabfb0eeb4...,8cd35b1e27159168...,ec78bc59e60cf5d6...,válida
5,5,validacion_reporte,trainer_script,2026-06-24 12:30:19,91d56bfabfb0eeb4...,024fbde22bf4fe51...,ec78bc59e60cf5d6...,7b7ec2191b6d1071...,válida


## Bloque 6 — Verificación final con `assert`

**Qué hace:** ejecuta el código exacto que muestra la diapositiva 19: un `assert` que detiene el notebook si la cadena no es válida, seguido de una tabla reducida con las columnas `indice`, `timestamp` y `hash_previo`.

**Para qué sirve:** garantiza formalmente que la cadena está íntegra antes de presentar el resultado. Si alguien hubiera roto la cadena, el `assert` fallaría y el notebook se detendría aquí en lugar de mostrar datos inválidos.

In [7]:
assert validar_cadena(cadena)
df = pd.DataFrame([asdict(b) for b in cadena])
df[["indice", "timestamp", "hash_previo"]]

,indice,timestamp,hash_previo
0,0,1.782304e+09,0
1,1,1.782304e+09,eab9a39c75d7e0a3b8a7bba8a08b420deb2936e0bacccf...
2,2,1.782304e+09,325e36115ffaacfb33dc5b0ed6ff6489c8b52f7d1a8185...
3,3,1.782304e+09,e9141efb282dc074ef814f3f456a04754c147f19e3d7b7...
4,4,1.782304e+09,8cd35b1e271591686a56858881d18822d5ab648d125bb1...
5,5,1.782304e+09,ec78bc59e60cf5d68019a0311d63351264e464182d9248...


## Bloque 7 — Prueba de alteración (detección de la ruptura)

**Qué hace:** sobre una **copia** de la cadena (con `deepcopy`, para no dañar la original), altera el hash de salida del bloque 3 (features) y vuelve a validar. Compara el hash recalculado del bloque 3 con el `hash_previo` que guarda el bloque 4 y guarda la evidencia en un archivo de texto.

**Para qué sirve:** demuestra la propiedad central del encadenamiento. Al modificar un bloque intermedio, su hash cambia y deja de coincidir con la referencia que guarda el bloque siguiente, por lo que la validación detecta la manipulación e identifica el punto exacto de la ruptura.

In [8]:
# Prueba de alteración controlada
# Se modifica una copia de la cadena para no destruir la cadena original.
cadena_alterada = deepcopy(cadena)

print("VALIDACIÓN ANTES DE ALTERAR")
print("---------------------------")
print("¿La cadena alterada es válida antes del cambio?:", validar_cadena(cadena_alterada))

print("\n>>> Alterando el bloque 3: paso 'features'")
print("Se cambia manualmente el hash de salida del bloque de features.")

# Alteramos un bloque intermedio.
# El bloque 4 todavía conserva el hash original del bloque 3,
# por eso la validación debe detectar una ruptura.
cadena_alterada[3].datos["output"] = "hash_falso_manipulado_0000000000000000"

print("\nVALIDACIÓN DESPUÉS DE ALTERAR")
print("----------------------------")
resultado_alterado = validar_cadena(cadena_alterada)
print("¿La cadena alterada es válida después del cambio?:", resultado_alterado)

print("\nCOMPARACIÓN DE HASHES")
print("---------------------")
print(f"Hash recalculado del bloque 3: {hash_bloque(cadena_alterada[3])[:24]}...")
print(f"Hash previo guardado en bloque 4: {cadena_alterada[4].hash_previo[:24]}...")
print("Conclusión: los hashes ya no coinciden, por lo tanto se detecta la manipulación.")

# Guardamos evidencia de la prueba de alteración
with open("artefactos/prueba_alteracion.txt", "w", encoding="utf-8") as archivo:
    archivo.write("PRUEBA DE ALTERACIÓN CONTROLADA\n")
    archivo.write("================================\n\n")
    archivo.write("Se alteró manualmente el bloque 3, correspondiente al paso de features.\n")
    archivo.write("El bloque 4 conserva el hash previo original del bloque 3.\n\n")
    archivo.write(f"Cadena original válida: {validar_cadena(cadena)}\n")
    archivo.write(f"Cadena alterada válida: {resultado_alterado}\n\n")
    archivo.write(f"Hash recalculado del bloque 3 alterado: {hash_bloque(cadena_alterada[3])}\n")
    archivo.write(f"Hash previo guardado en bloque 4: {cadena_alterada[4].hash_previo}\n")

print("\nArchivo generado: artefactos/prueba_alteracion.txt")

VALIDACIÓN ANTES DE ALTERAR
---------------------------
¿La cadena alterada es válida antes del cambio?: True

>>> Alterando el bloque 3: paso 'features'
Se cambia manualmente el hash de salida del bloque de features.

VALIDACIÓN DESPUÉS DE ALTERAR
----------------------------
  ✗ Ruptura detectada en el bloque 4 (paso: entrenamiento)
¿La cadena alterada es válida después del cambio?: False

COMPARACIÓN DE HASHES
---------------------
Hash recalculado del bloque 3: cfea3fb400be64f9d5fcf7c8...
Hash previo guardado en bloque 4: 8cd35b1e271591686a568588...
Conclusión: los hashes ya no coinciden, por lo tanto se detecta la manipulación.

Archivo generado: artefactos/prueba_alteracion.txt


## Conclusión crítica

El notebook implementa una cadena de bloques didáctica para auditar un pipeline de Machine Learning usando el dataset Iris. Cada etapa del proceso queda registrada como un bloque: ingesta del dataset bruto, limpieza, generación de características, entrenamiento del modelo y validación mediante métricas.

El uso de hashes permite dejar evidencia verificable de los artefactos generados en cada etapa. Además, el campo `hash_previo` enlaza cada bloque con el anterior, por lo que una alteración en un bloque intermedio modifica su hash y rompe la consistencia de los bloques posteriores.

La prueba de alteración demuestra que, al modificar el bloque de features, la validación detecta una ruptura en el bloque siguiente. Esto muestra que la cadena permite detectar cambios posteriores en la historia registrada del pipeline.

Este diseño aporta valor cuando se requiere trazabilidad, auditoría y control de versiones en procesos de datos compartidos por varios actores. Sin embargo, puede ser sobreingeniería para trabajos pequeños, locales o individuales, donde podría bastar con Git, logs firmados o una base de datos auditable.

Finalmente, esta implementación no demuestra que los datos originales sean verdaderos; solo demuestra que la representación registrada no fue modificada sin dejar evidencia.